In [ ]:
from langchain_core.documents import Document

## State 정의

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator

# state 정의
class GraphState(TypedDict):
    context: Annotated[Sequence[Document], operator.add]
    answer: Annotated[Sequence[Document], operator.add]
    question: Annotated[str, operator.add]
    sql_query: Annotated[str, operator.add]
    binary_score: Annotated[str, operator.add]

## Node 정의

In [ ]:
def retrieve(state: GraphState) -> GraphState:
    # retrieve: 검색
    documents = ["검색된 문서"]
    return GraphState(context=state.get("context", []) + documents)

def rewrite_query(state: GraphState) -> GraphState:
    # Query Transform: 쿼리 재작성
    documents = ["검색된 문서"]
    return GraphState(context=state.get("context", []) + documents)

def llm_gpt_execute(state: GraphState) -> GraphState:
    # LLM 실행
    answer = ["GPT로 생성된 답변"]
    return GraphState(answer=state.get("answer", []) + answer)

def llm_claude_execute(state: GraphState) -> GraphState:
    # LLM 실행
    answer = ["Claude로 생성된 답변"]
    return GraphState(answer=state.get("answer", []) + answer)

def relevance_check(state: GraphState) -> GraphState:
    # Relevance Check: 관련성 확인
    binary_score = ["Relevance Score"]
    return GraphState(binary_score=state.get("binary_score", []) + binary_score)

def sum_up(state: GraphState) -> GraphState:
    # 결과 종합
    context = ["Result"]
    return GraphState(context=state.get("context", []) + context)

def validate_sql_query(state: GraphState) -> GraphState:
    # validate SQL Query: SQL 쿼리 검증
    binary_score = ["SQL 쿼리 검증 결과"]
    return GraphState(binary_score=state.get("binary_score", []) + binary_score)

def handle_error(state: GraphState) -> GraphState:
    # Error Handling: 에러 처리
    error = ["에러 발생"]
    return GraphState(context=state.get("context", []) + error)

def decision(state: GraphState) -> str:
    # 의사 결정
    # 예제 의사결정 로직: context의 길이에 따라 결정
    if len(state.get("context", [])) > 5:
        return "종료"
    else:
        return "재검색"

## 그래프 정의

In [ ]:
from IPython.display import Image, display
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

# (1): Conventional RAG
# (2): 재검색
# (3): 멀티 LLM
# (4): 쿼리 재작성

# langgraph.graph에서 StateGraph와 END를 가져옵니다.
workflow = StateGraph(GraphState)

# 노드를 추가합니다.
workflow.add_node("retrieve", retrieve)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("GPT 요청", llm_gpt_execute)
workflow.add_node("Claude 요청", llm_claude_execute)
workflow.add_node("GPT relevance check", relevance_check)
workflow.add_node("Claude relevance check", relevance_check)
workflow.add_node("결과 종합", sum_up)

# 노드를 연결합니다.
workflow.add_edge("retrieve", "GPT 요청")
workflow.add_edge("retrieve", "Claude 요청") # (3)
workflow.add_edge("rewrite_query", "retrieve") # (4)
workflow.add_edge("GPT 요청", "GPT relevance check")
workflow.add_edge("GPT relevance check", "결과 종합")
workflow.add_edge("Claude 요청", "Claude relevance check") # (3)
workflow.add_edge("Claude relevance check", "결과 종합") # (3)
workflow.add_edge("결과 종합", sum_up)

# 조건부 엣지를 추가합니다. (4)
workflow.add_conditional_edges(
    "결과 종합", # 관련성 체크 노드에서 나온 결과를 is_relevant 함수에 전달합니다.
    decision,
    {
        "재검색": "rewrite_query", # 관련성이 있으면 종료합니다.
        "종료": END # 관련성 체크 결과가 모호하다면 다시 답변을 생성합니다.
    }
)

# 시작점을 설정합니다.
workflow.set_entry_point("retrieve")

# 기록을 위한 메모리 저장소를 설정합니다.
memory = MemorySaver()

# 그래프를 컴파일합니다.
app = workflow.compile(checkpointer=memory)

try:
    display(
        Image(app.get_graph(xray=True).draw_mermaid_png())
    ) # 실행 가능한 객체의 그래프를 mermaid 형식의 PNG로 그려서 표시합니다.
    # xray=True는 추가적인 세부 정보를 포함합니다.
except:
    # 이 부분은 추가적인 의존성이 필요하며 선택적으로 실행됩니다.
    pass